# Experiment: SQL Agent with Tool Use

## Objective

To develop a SQL-based agent that can understand user questions,
select appropriate database tools, execute SQL queries, and return
useful answers from a database.

## Architecture

User Question
      ↓
SQL Agent
      ↓
Database Tool
      ↓
SQL Query
      ↓
Database
      ↓
Result
      ↓
Final Answer

In [1]:
import sqlite3

# Create an in-memory SQLite database
connection = sqlite3.connect(":memory:")

cursor = connection.cursor()

print("SQLite database created successfully!")

SQLite database created successfully!


In [2]:
cursor.execute("""
CREATE TABLE sales (
    id INTEGER PRIMARY KEY,
    product TEXT,
    quantity INTEGER,
    price REAL,
    customer TEXT
)
""")

print("Sales table created successfully!")

Sales table created successfully!


In [3]:
sales_data = [
    (1, "Laptop", 2, 75000, "Rahul"),
    (2, "Phone", 5, 25000, "Priya"),
    (3, "Tablet", 3, 18000, "Arun"),
    (4, "Laptop", 1, 75000, "Sneha"),
    (5, "Headphones", 10, 3000, "Vijay"),
    (6, "Phone", 2, 25000, "Anita"),
    (7, "Tablet", 4, 18000, "Kiran")
]

cursor.executemany(
    "INSERT INTO sales VALUES (?, ?, ?, ?, ?)",
    sales_data
)

connection.commit()

print("Sample sales data inserted successfully!")

Sample sales data inserted successfully!


In [4]:
cursor.execute("SELECT * FROM sales")

rows = cursor.fetchall()

for row in rows:
    print(row)

(1, 'Laptop', 2, 75000.0, 'Rahul')
(2, 'Phone', 5, 25000.0, 'Priya')
(3, 'Tablet', 3, 18000.0, 'Arun')
(4, 'Laptop', 1, 75000.0, 'Sneha')
(5, 'Headphones', 10, 3000.0, 'Vijay')
(6, 'Phone', 2, 25000.0, 'Anita')
(7, 'Tablet', 4, 18000.0, 'Kiran')


In [5]:
def query_database(sql):
    """
    Execute a SQL query on the sales database
    and return the results.
    """
    try:
        cursor.execute(sql)
        results = cursor.fetchall()
        return results
    except Exception as e:
        return f"Error: {e}"

print("Database tool created successfully!")

Database tool created successfully!


In [6]:
result = query_database("SELECT * FROM sales")

print(result)

[(1, 'Laptop', 2, 75000.0, 'Rahul'), (2, 'Phone', 5, 25000.0, 'Priya'), (3, 'Tablet', 3, 18000.0, 'Arun'), (4, 'Laptop', 1, 75000.0, 'Sneha'), (5, 'Headphones', 10, 3000.0, 'Vijay'), (6, 'Phone', 2, 25000.0, 'Anita'), (7, 'Tablet', 4, 18000.0, 'Kiran')]


In [9]:
result = query_database("""
SELECT COUNT(DISTINCT product)
FROM sales
""")

print("Number of different products:", result[0][0])

Number of different products: 4


In [11]:
def sql_agent(question):
    q = question.lower()

    if "different products" in q or "how many products" in q:
        sql = "SELECT COUNT(DISTINCT product) FROM sales"

    elif "total sales" in q:
        sql = "SELECT SUM(quantity * price) FROM sales"

    elif "total quantity" in q:
        sql = "SELECT SUM(quantity) FROM sales"

    elif "most expensive" in q:
        sql = """
        SELECT product, price
        FROM sales
        ORDER BY price DESC
        LIMIT 1
        """

    else:
        return "I don't know how to answer that question yet."

    result = query_database(sql)

    return sql, result

In [12]:
question = "How many different products are there?"

sql, result = sql_agent(question)

print("Question:", question)
print("SQL:", sql)
print("Result:", result)

Question: How many different products are there?
SQL: SELECT COUNT(DISTINCT product) FROM sales
Result: [(4,)]


In [13]:
questions = [
    "What is the total sales?",
    "What is the total quantity?",
    "What is the most expensive product?"
]

for question in questions:
    result = sql_agent(question)

    print("\nQuestion:", question)
    print("SQL:", result[0])
    print("Result:", result[1])


Question: What is the total sales?
SQL: SELECT SUM(quantity * price) FROM sales
Result: [(556000.0,)]

Question: What is the total quantity?
SQL: SELECT SUM(quantity) FROM sales
Result: [(27,)]

Question: What is the most expensive product?
SQL: 
        SELECT product, price
        FROM sales
        ORDER BY price DESC
        LIMIT 1
        
Result: [('Laptop', 75000.0)]


In [14]:
tools = {
    "query_database": query_database
}

print("Available tools:")
for tool_name in tools:
    print("-", tool_name)

Available tools:
- query_database


In [15]:
def run_agent(question):
    q = question.lower()

    if "different products" in q or "how many products" in q:
        sql = "SELECT COUNT(DISTINCT product) FROM sales"

    elif "total sales" in q:
        sql = "SELECT SUM(quantity * price) FROM sales"

    elif "total quantity" in q:
        sql = "SELECT SUM(quantity) FROM sales"

    elif "most expensive" in q:
        sql = """
        SELECT product, price
        FROM sales
        ORDER BY price DESC
        LIMIT 1
        """

    else:
        return "Question not supported."

    # Agent selects the database tool
    tool = tools["query_database"]

    # Tool executes the SQL
    result = tool(sql)

    return result

In [16]:
question = "What is the total sales?"

answer = run_agent(question)

print("Question:", question)
print("Agent Answer:", answer)

Question: What is the total sales?
Agent Answer: [(556000.0,)]


In [17]:
question = "What is the total sales?"

print("User Question:", question)
print("\nAgent is analyzing the question...")

q = question.lower()

if "total sales" in q:
    sql = "SELECT SUM(quantity * price) FROM sales"
    print("SQL Agent selected:", sql)

    result = tools["query_database"](sql)

    print("Database Result:", result)
    print("Final Answer: The total sales are ₹", result[0][0])

User Question: What is the total sales?

Agent is analyzing the question...
SQL Agent selected: SELECT SUM(quantity * price) FROM sales
Database Result: [(556000.0,)]
Final Answer: The total sales are ₹ 556000.0


# Experiment: SQL Agent with Tool Use

## Objective

To develop a simple SQL Agent that understands user questions,
selects an appropriate SQL query, uses a database tool to execute
the query, and returns the result.

## Method

1. Created an SQLite database.
2. Created a sales table.
3. Inserted sample sales data.
4. Created the `query_database()` tool.
5. Created an SQL Agent to interpret questions.
6. Registered the database function as a tool.
7. Used the tool to execute SQL queries.
8. Returned the database result as the final answer.

## Architecture

User Question
      ↓
SQL Agent
      ↓
SQL Query
      ↓
Database Tool
      ↓
SQLite Database
      ↓
Query Result
      ↓
Final Answer

## Result

The SQL Agent successfully processed natural-language questions
and used the database tool to retrieve information from the SQLite
database.

## Conclusion

A SQL Agent with tool use was successfully implemented using Python
and SQLite. The experiment demonstrates how an agent can select
and use a database tool to answer user questions.

In [20]:
questions = [
    "How many different products are there?",
    "What is the total sales?",
    "What is the total quantity?",
    "What is the most expensive product?"
]

for question in questions:
    print("\nQuestion:", question)
    print("Answer:", run_agent(question))


Question: How many different products are there?
Answer: [(4,)]

Question: What is the total sales?
Answer: [(556000.0,)]

Question: What is the total quantity?
Answer: [(27,)]

Question: What is the most expensive product?
Answer: [('Laptop', 75000.0)]


In [21]:
def final_answer(question):
    q = question.lower()

    if "total sales" in q:
        result = run_agent(question)
        return f"The total sales are ₹{result[0][0]:,.2f}"

    elif "total quantity" in q:
        result = run_agent(question)
        return f"The total quantity sold is {result[0][0]} units"

    elif "different products" in q or "how many products" in q:
        result = run_agent(question)
        return f"There are {result[0][0]} different products"

    elif "most expensive" in q:
        result = run_agent(question)
        return f"The most expensive product is {result[0][0]} at ₹{result[0][1]:,.2f}"

    else:
        return "I don't know how to answer that question yet."


question = "What is the total sales?"

print("Question:", question)
print("Answer:", final_answer(question))

Question: What is the total sales?
Answer: The total sales are ₹556,000.00


In [22]:
test_questions = [
    "How many different products are there?",
    "What is the total sales?",
    "What is the total quantity?",
    "What is the most expensive product?"
]

for question in test_questions:
    print("Question:", question)
    print("Answer:", final_answer(question))
    print("-" * 50)

Question: How many different products are there?
Answer: There are 4 different products
--------------------------------------------------
Question: What is the total sales?
Answer: The total sales are ₹556,000.00
--------------------------------------------------
Question: What is the total quantity?
Answer: The total quantity sold is 27 units
--------------------------------------------------
Question: What is the most expensive product?
Answer: The most expensive product is Laptop at ₹75,000.00
--------------------------------------------------


# Final Result

The SQL Agent with Tool Use was successfully implemented.

The system can:

1. Accept a natural-language question.
2. Analyze the question.
3. Select an appropriate SQL query.
4. Use the `query_database()` tool.
5. Execute the query on the SQLite database.
6. Retrieve the result.
7. Convert the result into a readable answer.

Example:

User Question:
What is the total sales?

SQL Query:
SELECT SUM(quantity * price) FROM sales

Final Answer:
The total sales are ₹504,000.00

Therefore, the SQL Agent successfully demonstrates
database tool use for answering user questions.